# LeetCode 704 — Binary Search bằng Python

**Link đề:** https://leetcode.com/problems/binary-search/
**Độ khó:** Easy · **Chủ đề:** Array, Binary Search

```python
class Solution:
    def search(self, nums: List[int], target: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu vì sao mảng **đã sắp xếp** cho phép tìm kiếm `O(log n)` thay vì `O(n)`.
2. Nắm chắc **một khuôn binary search** không lỗi: khoảng đóng `[lo, hi]`, điều kiện `lo <= hi`, cập nhật `mid ± 1`.
3. Biết khuôn thứ hai: khoảng nửa mở `[lo, hi)` và hàm `bisect_left` có sẵn.
4. Trace từng bước `lo, hi, mid`, chứng minh bằng invariant, đếm số vòng `≈ log₂ n`.
5. Tránh các lỗi kinh điển: vòng lặp vô hạn, sai `<`/`<=`, quên `+1/-1`, và áp dụng sang bài "lower bound".

## 1. Đề bài và ví dụ

Cho mảng `nums` **tăng dần** và số `target`. Nếu `target` có trong `nums`, trả về **chỉ số** của nó; nếu không, trả `-1`.

**Bắt buộc:** thuật toán `O(log n)`.

| `nums` | `target` | Output | Giải thích |
|---|:---:|:---:|---|
| `[-1,0,3,5,9,12]` | `9` | `4` | `nums[4] = 9` |
| `[-1,0,3,5,9,12]` | `2` | `-1` | không có 2 |

**Constraints:** `1 <= len(nums) <= 10^4`; `-10^4 < nums[i], target < 10^4`; các phần tử **khác nhau**; `nums` tăng dần.

In [1]:
nums = [-1, 0, 3, 5, 9, 12]
for i, x in enumerate(nums):
    print(f"chỉ số {i}: {x}")

chỉ số 0: -1
chỉ số 1: 0
chỉ số 2: 3
chỉ số 3: 5
chỉ số 4: 9
chỉ số 5: 12


## 2. Nền tảng Python

### 2.1. Chỉ số, `len`, chỉ số cuối là `len - 1`

In [2]:
nums = [-1, 0, 3, 5, 9, 12]
print("len =", len(nums), "| chỉ số đầu = 0 | chỉ số cuối =", len(nums) - 1, "->", nums[len(nums) - 1])

len = 6 | chỉ số đầu = 0 | chỉ số cuối = 5 -> 12


### 2.2. Chia lấy phần nguyên `//`

`(lo + hi) // 2` cho **điểm giữa làm tròn xuống**. `/` cho số thực (không dùng làm chỉ số được).

In [3]:
print(7 / 2, 7 // 2)          # 3.5  3
print((0 + 5) // 2, (2 + 3) // 2, (4 + 4) // 2)
try:
    print(nums[5 / 2])
except TypeError as e:
    print("Lỗi:", e)

3.5 3
2 2 4
Lỗi: list indices must be integers or slices, not float


### 2.3. Vòng lặp `while`

`while điều_kiện:` lặp **chừng nào** điều kiện còn đúng. Binary search dùng `while` vì ta không biết trước số vòng, chỉ biết dừng khi khoảng tìm kiếm rỗng hoặc tìm thấy.

### 2.4. `log₂ n` — số lần chia đôi

Mỗi lần chia đôi, kích thước khoảng giảm một nửa. Từ `n` về `1` mất khoảng `log₂ n` lần.

In [4]:
import math
for n in [8, 1000, 10**4, 10**6, 10**9]:
    steps, size = 0, n
    while size > 1:
        size //= 2
        steps += 1
    print(f"n = {n:>10}: chia đôi {steps:2} lần | log2(n) ≈ {math.log2(n):.1f}")

n =          8: chia đôi  3 lần | log2(n) ≈ 3.0
n =       1000: chia đôi  9 lần | log2(n) ≈ 10.0
n =      10000: chia đôi 13 lần | log2(n) ≈ 13.3
n =    1000000: chia đôi 19 lần | log2(n) ≈ 19.9
n = 1000000000: chia đôi 29 lần | log2(n) ≈ 29.9


## 3. Cách 1 — Tìm tuần tự (linear search)

Duyệt từ đầu đến cuối. Đúng, nhưng `O(n)` — **không đạt yêu cầu** `O(log n)` của đề và không dùng tới tính chất "đã sắp xếp".

In [5]:
def linear_search(nums, target):
    for i, x in enumerate(nums):
        if x == target:
            return i
    return -1

print(linear_search([-1, 0, 3, 5, 9, 12], 9))   # 4
print(linear_search([-1, 0, 3, 5, 9, 12], 2))   # -1

4
-1


Cải tiến nhỏ: vì mảng tăng dần, gặp `x > target` thì dừng sớm. Vẫn `O(n)` trường hợp xấu (target lớn).

## 4. Nút thắt và insight

Trò chơi "đoán số từ 1 đến 100": người ta nói "lớn hơn / nhỏ hơn". Chiến lược tốt nhất là **đoán giữa**: mỗi câu trả lời loại **một nửa** số khả năng.

Với mảng tăng dần, so `target` với phần tử giữa `nums[mid]`:
- `nums[mid] == target` → tìm thấy.
- `nums[mid] < target` → mọi phần tử **từ đầu tới `mid`** đều `< target` → bỏ nửa trái, tìm trong `[mid+1, hi]`.
- `nums[mid] > target` → mọi phần tử **từ `mid` tới cuối** đều `> target` → bỏ nửa phải, tìm trong `[lo, mid-1]`.

In [6]:
nums = [-1, 0, 3, 5, 9, 12]
target = 9
lo, hi = 0, len(nums) - 1
mid = (lo + hi) // 2
print("mid =", mid, "nums[mid] =", nums[mid], "| so với target", target, "->",
      "bỏ nửa trái" if nums[mid] < target else "bỏ nửa phải" if nums[mid] > target else "thấy")

mid = 2 nums[mid] = 3 | so với target 9 -> bỏ nửa trái


## 5. Cách 2 — Binary search khoảng đóng `[lo, hi]` (lời giải chính)

Mổ xẻ từng dòng:

```python
lo, hi = 0, len(nums) - 1     # khoảng tìm kiếm [lo, hi], CẢ HAI ĐẦU đều còn là ứng viên
while lo <= hi:               # khoảng khác rỗng (lo == hi vẫn còn 1 phần tử)
    mid = (lo + hi) // 2
    if nums[mid] == target: return mid
    if nums[mid] < target:  lo = mid + 1   # mid đã xét, loại luôn mid
    else:                   hi = mid - 1
return -1                      # lo > hi: khoảng rỗng
```

Ba quyết định phải **khớp nhau**: khoảng đóng ⇔ `hi = len - 1` ⇔ `while lo <= hi` ⇔ `mid ± 1`.

In [7]:
def binary_search_closed(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

print(binary_search_closed([-1, 0, 3, 5, 9, 12], 9))   # 4
print(binary_search_closed([-1, 0, 3, 5, 9, 12], 2))   # -1
print(binary_search_closed([5], 5), binary_search_closed([5], 7))

4
-1
0 -1


## 6. TRACE từng bước

**Ví dụ 1:** `nums = [-1, 0, 3, 5, 9, 12]`, `target = 9`

| Vòng | lo | hi | mid | nums[mid] | So sánh | Hành động |
|---|:---:|:---:|:---:|:---:|---|---|
| 1 | 0 | 5 | 2 | 3 | 3 < 9 | lo = 3 |
| 2 | 3 | 5 | 4 | 9 | 9 == 9 | **trả 4** |

**Ví dụ 2:** `target = 2`

| Vòng | lo | hi | mid | nums[mid] | So sánh | Hành động |
|---|:---:|:---:|:---:|:---:|---|---|
| 1 | 0 | 5 | 2 | 3 | 3 > 2 | hi = 1 |
| 2 | 0 | 1 | 0 | -1 | -1 < 2 | lo = 1 |
| 3 | 1 | 1 | 1 | 0 | 0 < 2 | lo = 2 |
| dừng | 2 | 1 | | | lo > hi | **trả -1** |

Để ý vòng 3: `lo == hi` vẫn phải kiểm tra — đó là lý do điều kiện là `<=`.

In [8]:
def trace_binary_search(nums, target):
    lo, hi = 0, len(nums) - 1
    step = 1
    while lo <= hi:
        mid = (lo + hi) // 2
        v = nums[mid]
        if v == target:
            print(f"vòng {step}: lo={lo}, hi={hi}, mid={mid}, nums[mid]={v} == {target} -> trả {mid}")
            return mid
        if v < target:
            print(f"vòng {step}: lo={lo}, hi={hi}, mid={mid}, nums[mid]={v} < {target} -> lo = {mid + 1}")
            lo = mid + 1
        else:
            print(f"vòng {step}: lo={lo}, hi={hi}, mid={mid}, nums[mid]={v} > {target} -> hi = {mid - 1}")
            hi = mid - 1
        step += 1
    print(f"dừng: lo={lo} > hi={hi} -> trả -1")
    return -1

trace_binary_search([-1, 0, 3, 5, 9, 12], 9)
print()
trace_binary_search([-1, 0, 3, 5, 9, 12], 2)

vòng 1: lo=0, hi=5, mid=2, nums[mid]=3 < 9 -> lo = 3
vòng 2: lo=3, hi=5, mid=4, nums[mid]=9 == 9 -> trả 4

vòng 1: lo=0, hi=5, mid=2, nums[mid]=3 > 2 -> hi = 1
vòng 2: lo=0, hi=1, mid=0, nums[mid]=-1 < 2 -> lo = 1
vòng 3: lo=1, hi=1, mid=1, nums[mid]=0 < 2 -> lo = 2
dừng: lo=2 > hi=1 -> trả -1


-1

Trace với mảng dài hơn để thấy số vòng nhỏ (16 phần tử, tối đa 5 vòng):

In [9]:
big = list(range(0, 32, 2))   # [0, 2, 4, ..., 30]
print(big)
trace_binary_search(big, 22)
print()
trace_binary_search(big, 31)

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
vòng 1: lo=0, hi=15, mid=7, nums[mid]=14 < 22 -> lo = 8
vòng 2: lo=8, hi=15, mid=11, nums[mid]=22 == 22 -> trả 11

vòng 1: lo=0, hi=15, mid=7, nums[mid]=14 < 31 -> lo = 8
vòng 2: lo=8, hi=15, mid=11, nums[mid]=22 < 31 -> lo = 12
vòng 3: lo=12, hi=15, mid=13, nums[mid]=26 < 31 -> lo = 14
vòng 4: lo=14, hi=15, mid=14, nums[mid]=28 < 31 -> lo = 15
vòng 5: lo=15, hi=15, mid=15, nums[mid]=30 < 31 -> lo = 16
dừng: lo=16 > hi=15 -> trả -1


-1

## 7. Code nộp LeetCode

In [10]:
from typing import List

class Solution:
    def search(self, nums: List[int], target: int) -> int:
        lo, hi = 0, len(nums) - 1
        while lo <= hi:
            mid = (lo + hi) // 2
            if nums[mid] == target:
                return mid
            if nums[mid] < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1

sol = Solution()
print(sol.search([-1, 0, 3, 5, 9, 12], 9))   # 4
print(sol.search([-1, 0, 3, 5, 9, 12], 2))   # -1

4
-1


**Về tràn số:** trong C++/Java, `(lo + hi) / 2` có thể tràn `int` khi `lo + hi > 2^31 - 1`, nên người ta viết `lo + (hi - lo) // 2`. Python có số nguyên không giới hạn nên không tràn, nhưng nên biết cách viết này khi phỏng vấn.

## 8. Cách 3 — Khoảng nửa mở `[lo, hi)`

Khuôn thứ hai, rất phổ biến vì dễ mở rộng sang "lower bound":
- `hi = len(nums)` (hi **không** phải ứng viên).
- `while lo < hi` (khoảng rỗng khi `lo == hi`).
- Khi `nums[mid] > target`: `hi = mid` (không trừ 1, vì `hi` vốn không thuộc khoảng).

In [11]:
def binary_search_half_open(nums, target):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return -1

print(binary_search_half_open([-1, 0, 3, 5, 9, 12], 9), binary_search_half_open([-1, 0, 3, 5, 9, 12], 2))

4 -1


### 8.1. Biến dạng "lower bound": tìm vị trí đầu tiên `>= target`

Không dừng khi bằng; luôn thu hẹp tới khi `lo == hi`. Kết quả `lo` là **vị trí chèn**. Sau đó kiểm tra `nums[lo] == target`.

In [12]:
def lower_bound(nums, target):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo

def binary_search_lb(nums, target):
    i = lower_bound(nums, target)
    return i if i < len(nums) and nums[i] == target else -1

nums = [-1, 0, 3, 5, 9, 12]
for t in [-5, 2, 3, 12, 20]:
    print(f"target={t:>3}: lower_bound={lower_bound(nums, t)}, search={binary_search_lb(nums, t)}")

target= -5: lower_bound=0, search=-1
target=  2: lower_bound=2, search=-1
target=  3: lower_bound=2, search=2
target= 12: lower_bound=5, search=5
target= 20: lower_bound=6, search=-1


Chú ý `i < len(nums) and nums[i] == target`: khi `target` lớn hơn mọi phần tử, `i = len(nums)` — phải kiểm `i < len(nums)` **trước** để không lỗi chỉ số (short-circuit).

## 9. Cách 4 — Dùng thư viện `bisect`

`bisect.bisect_left(nums, target)` chính là `lower_bound` ở trên, viết bằng C nên nhanh.

In [13]:
import bisect

def binary_search_bisect(nums, target):
    i = bisect.bisect_left(nums, target)
    return i if i < len(nums) and nums[i] == target else -1

print(bisect.bisect_left([1, 3, 5], 3), bisect.bisect_right([1, 3, 5], 3), bisect.bisect_left([1, 3, 5], 4))
print(binary_search_bisect([-1, 0, 3, 5, 9, 12], 9), binary_search_bisect([-1, 0, 3, 5, 9, 12], 2))

1 2 2
4 -1


## 10. Cách 5 — Đệ quy

Cùng ý tưởng, thay vòng lặp bằng gọi hàm trên nửa còn lại. Tốn thêm `O(log n)` bộ nhớ call stack.

In [14]:
def binary_search_rec(nums, target, lo=0, hi=None):
    if hi is None:
        hi = len(nums) - 1
    if lo > hi:
        return -1
    mid = (lo + hi) // 2
    if nums[mid] == target:
        return mid
    if nums[mid] < target:
        return binary_search_rec(nums, target, mid + 1, hi)
    return binary_search_rec(nums, target, lo, mid - 1)

print(binary_search_rec([-1, 0, 3, 5, 9, 12], 9), binary_search_rec([-1, 0, 3, 5, 9, 12], 2))

4 -1


`hi=None` là **tham số mặc định**: người gọi không cần truyền; lần đầu ta tự đặt `hi = len(nums) - 1`. Không đặt mặc định `hi=len(nums)-1` trực tiếp được vì giá trị mặc định tính **một lần lúc định nghĩa hàm**, khi đó chưa có `nums`.

## 11. Vì sao đúng?

**Invariant (khoảng đóng):** *Nếu `target` có trong `nums` thì chỉ số của nó nằm trong `[lo, hi]`.*

- Ban đầu `[0, n-1]` là cả mảng → đúng.
- `nums[mid] < target`: do mảng tăng, mọi `nums[i]` với `i <= mid` đều `<= nums[mid] < target` → target không ở `[lo, mid]` → đặt `lo = mid + 1` giữ invariant.
- `nums[mid] > target`: tương tự, loại `[mid, hi]` → `hi = mid - 1`.
- Nếu trả `mid` thì hiển nhiên đúng. Nếu vòng lặp kết thúc với `lo > hi`, khoảng rỗng → theo invariant, target không có → `-1` đúng.

**Kết thúc:** mỗi vòng, `hi - lo` giảm thực sự (vì `mid` nằm trong `[lo, hi]` và ta loại ít nhất `mid`), nên vòng lặp dừng. Kích thước giảm ít nhất một nửa → tối đa `⌊log₂ n⌋ + 1` vòng.

In [15]:
def count_iterations(nums, target):
    lo, hi, it = 0, len(nums) - 1, 0
    while lo <= hi:
        it += 1
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return it
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return it

n = 10**4
arr = list(range(n))
worst = max(count_iterations(arr, t) for t in range(-1, n + 1))
print("n =", n, "| số vòng tối đa =", worst, "| floor(log2 n) + 1 =", math.floor(math.log2(n)) + 1)

n = 10000 | số vòng tối đa = 14 | floor(log2 n) + 1 = 14


## 12. Độ phức tạp và so sánh

Gọi `n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Linear search | `O(n)` | `O(1)` | Không đạt yêu cầu đề |
| 2. Khoảng đóng `[lo, hi]` | `O(log n)` | `O(1)` | **Nên nộp**, trực quan |
| 3. Nửa mở `[lo, hi)` / lower bound | `O(log n)` | `O(1)` | Dễ mở rộng cho bài "vị trí đầu tiên" |
| 4. `bisect_left` | `O(log n)` | `O(1)` | Ngắn, nhanh; cần hiểu bản tự viết trước |
| 5. Đệ quy | `O(log n)` | `O(log n)` | Call stack |

Với `n = 10^4`: linear tối đa 10000 phép so sánh, binary tối đa 14.

## 13. Bộ test — mẫu, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:30} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.search([-1, 0, 3, 5, 9, 12], 9), 4)
check("mẫu 2", sol.search([-1, 0, 3, 5, 9, 12], 2), -1)
check("1 phần tử, có", sol.search([5], 5), 0)
check("1 phần tử, không", sol.search([5], -5), -1)
check("phần tử đầu", sol.search([-1, 0, 3, 5, 9, 12], -1), 0)
check("phần tử cuối", sol.search([-1, 0, 3, 5, 9, 12], 12), 5)
check("nhỏ hơn tất cả", sol.search([-1, 0, 3, 5, 9, 12], -9999), -1)
check("lớn hơn tất cả", sol.search([-1, 0, 3, 5, 9, 12], 9999), -1)
check("2 phần tử, lấy cái sau", sol.search([1, 2], 2), 1)
check("số âm", sol.search([-9, -5, -2], -5), 1)

mẫu 1                          PASS (4)
mẫu 2                          PASS (-1)
1 phần tử, có                  PASS (0)
1 phần tử, không               PASS (-1)
phần tử đầu                    PASS (0)
phần tử cuối                   PASS (5)
nhỏ hơn tất cả                 PASS (-1)
lớn hơn tất cả                 PASS (-1)
2 phần tử, lấy cái sau         PASS (1)
số âm                          PASS (1)


In [17]:
import random
rng = random.Random(704)
funcs = [linear_search, binary_search_closed, binary_search_half_open, binary_search_lb,
         binary_search_bisect, binary_search_rec, sol.search]
for _ in range(500):
    arr = sorted(rng.sample(range(-60, 60), rng.randint(1, 30)))
    for t in range(-62, 62):
        exp = arr.index(t) if t in arr else -1
        for f in funcs:
            assert f(arr, t) == exp, (f, arr, t)
print("500 mảng × 124 target: 7 cách đều khớp")

500 mảng × 124 target: 7 cách đều khớp


## 14. Lỗi người mới hay gặp

### Lỗi 1 — `while lo < hi` với khoảng đóng (bỏ sót phần tử cuối cùng)

In [18]:
def wrong_lt(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo < hi:                       # SAI với khoảng đóng
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

print("sai:", wrong_lt([5], 5), "| đúng:", sol.search([5], 5))
print("sai:", wrong_lt([1, 3], 3), "| đúng:", sol.search([1, 3], 3))

sai: -1 | đúng: 0
sai: -1 | đúng: 1


Vì sao sai: khi `lo == hi` vẫn còn **một** ứng viên chưa kiểm tra. Sửa: `while lo <= hi`.

### Lỗi 2 — `lo = mid` (không +1) → vòng lặp vô hạn

Với `lo = 0, hi = 1`: `mid = 0`; nếu `nums[0] < target` mà gán `lo = mid = 0` thì không có gì thay đổi → lặp mãi. Ta giới hạn số vòng để minh họa:

In [19]:
def wrong_lo_mid(nums, target, max_iter=20):
    lo, hi = 0, len(nums) - 1
    it = 0
    while lo <= hi:
        it += 1
        if it > max_iter:
            return f"VÒNG LẶP VÔ HẠN (đã dừng sau {max_iter} vòng, lo={lo}, hi={hi})"
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid                     # SAI: quên + 1
        else:
            hi = mid - 1
    return -1

print(wrong_lo_mid([1, 3], 4))

VÒNG LẶP VÔ HẠN (đã dừng sau 20 vòng, lo=0, hi=1)


Vì sao sai: `mid` đã được kiểm tra, không loại nó thì khoảng có thể không co lại. Sửa: `lo = mid + 1` (khoảng đóng).

### Lỗi 3 — Trộn khuôn: `hi = len(nums)` nhưng `while lo <= hi`

In [20]:
def wrong_mixed(nums, target):
    lo, hi = 0, len(nums)                # khoảng nửa mở...
    while lo <= hi:                      # ...nhưng điều kiện của khoảng đóng
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

try:
    wrong_mixed([1, 3], 9)
except IndexError as e:
    print("Lỗi:", e)

Lỗi: list index out of range


Vì sao sai: `mid` có thể bằng `len(nums)` → `IndexError`. Sửa: chọn **một** khuôn và giữ nhất quán cả 3 chỗ (khởi tạo `hi`, điều kiện `while`, cập nhật `hi`).

### Lỗi 4 — Dùng `/` thay `//` → chỉ số kiểu `float` → `TypeError` (đã thấy ở mục 2.2).

### Lỗi 5 — Dùng `nums.index(target)` hay `target in nums`
Đúng kết quả nhưng là `O(n)` — vi phạm yêu cầu `O(log n)`. Trong phỏng vấn coi như chưa làm bài.

## 15. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu dùng binary search:**
- Dữ liệu **đã sắp xếp** (hoặc có tính đơn điệu: "nếu x thỏa thì mọi x lớn hơn cũng thỏa").
- Đề yêu cầu `O(log n)`.
- Tìm "vị trí đầu tiên / cuối cùng", "giá trị nhỏ nhất thỏa điều kiện" (binary search trên đáp án).

Bài liên quan: 35 Search Insert Position, 278 First Bad Version, 69 Sqrt(x), 34 First and Last Position, 744 Find Smallest Letter Greater Than Target, 33 Search in Rotated Sorted Array.

### 15.1. Biến thể A — Search Insert Position (35)
Chính là `lower_bound`.

In [21]:
print([lower_bound([1, 3, 5, 6], t) for t in [5, 2, 7, 0]])   # [2, 1, 4, 0]

[2, 1, 4, 0]


### 15.2. Biến thể B — binary search trên đáp án: căn bậc hai nguyên (69)
Tìm `r` lớn nhất với `r*r <= x`. Điều kiện `r*r <= x` **đơn điệu**: đúng với các `r` nhỏ, sai với `r` lớn.

In [22]:
def int_sqrt(x):
    lo, hi = 0, x
    ans = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        if mid * mid <= x:
            ans = mid          # mid thỏa, thử lớn hơn
            lo = mid + 1
        else:
            hi = mid - 1
    return ans

print([int_sqrt(x) for x in [0, 1, 4, 8, 15, 16, 2147395599]])

[0, 1, 2, 2, 3, 4, 46339]


## 16. Bài tập tự luyện

**Bài 1 (tính tay).** `nums = [1, 4, 7, 10, 13, 16, 19]`, `target = 16`. Ghi dãy giá trị `mid` (chỉ số) qua từng vòng (khuôn khoảng đóng) vào `ex1_mids`.

**Bài 2 (đoán output).** Với `nums = [2, 4, 6]`, `target = 5`, khuôn khoảng đóng kết thúc với `(lo, hi)` bằng bao nhiêu? Gán `ex2_lo_hi`.

**Bài 3 (TODO).** Tự viết `ex3_search(nums, target)` theo khuôn khoảng đóng.

**Bài 4 (biến thể).** `ex4_last_leq(nums, x)`: chỉ số **cuối cùng** có `nums[i] <= x` (hoặc `-1`).

**Bài 5 (biến thể).** `ex5_first_last(nums, target)`: LeetCode 34 — mảng **có trùng**, trả `[đầu, cuối]` hoặc `[-1, -1]`.

**Bài 6 (mini challenge).** `ex6_search_rotated(nums, target)`: LeetCode 33 — mảng tăng dần bị **xoay** (ví dụ `[4,5,6,7,0,1,2]`), phần tử khác nhau, yêu cầu `O(log n)`.

In [23]:
ex1_mids = None      # TODO
ex2_lo_hi = None     # TODO

def ex3_search(nums, target):
    # TODO
    pass

# assert ex1_mids == [...]
# assert ex3_search([-1, 0, 3, 5, 9, 12], 9) == 4

In [24]:
def ex4_last_leq(nums, x):
    # TODO
    pass

def ex5_first_last(nums, target):
    # TODO
    pass

def ex6_search_rotated(nums, target):
    # TODO
    pass

# assert ex4_last_leq([1, 3, 5], 4) == 1
# assert ex5_first_last([5, 7, 7, 8, 8, 10], 8) == [3, 4]
# assert ex6_search_rotated([4, 5, 6, 7, 0, 1, 2], 0) == 4

## 17. Gợi ý 3 mức

- **Mức 1:** Luôn viết invariant trước: "đáp án (nếu có) nằm trong `[lo, hi]`".
- **Mức 2:** Bài 4: khi `nums[mid] <= x`, `mid` là ứng viên → lưu `ans = mid`, đi phải. Bài 5: hai lần lower bound: `lower_bound(target)` và `lower_bound(target + 1) - 1`. Bài 6: tại mỗi `mid`, **ít nhất một nửa** `[lo, mid]` hoặc `[mid, hi]` là tăng dần; xác định nửa đó bằng `nums[lo] <= nums[mid]`.
- **Mức 3:** Bài 6:
  ```text
  if nums[lo] <= nums[mid]:            # nửa trái tăng dần
      if nums[lo] <= target < nums[mid]: hi = mid - 1
      else: lo = mid + 1
  else:                                # nửa phải tăng dần
      if nums[mid] < target <= nums[hi]: lo = mid + 1
      else: hi = mid - 1
  ```

## 18. Đáp án, template nhớ nhanh và checklist

Bài 1: `lo=0,hi=6 → mid=3 (10<16) → lo=4; mid=5 (16==16)` → `ex1_mids = [3, 5]`.
Bài 2: `mid=1 (4<5) → lo=2; mid=2 (6>5) → hi=1` → dừng với `(lo, hi) = (2, 1)`. Để ý `lo = 2` chính là vị trí chèn của 5.

In [25]:
ans_ex1_mids = [3, 5]
ans_ex2_lo_hi = (2, 1)

def ans_ex3_search(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

def ans_ex4_last_leq(nums, x):
    lo, hi, ans = 0, len(nums) - 1, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] <= x:
            ans = mid
            lo = mid + 1
        else:
            hi = mid - 1
    return ans

def ans_ex5_first_last(nums, target):
    a = lower_bound(nums, target)
    if a == len(nums) or nums[a] != target:
        return [-1, -1]
    return [a, lower_bound(nums, target + 1) - 1]

def ans_ex6_search_rotated(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1

def mids_closed(nums, target):
    lo, hi, out = 0, len(nums) - 1, []
    while lo <= hi:
        mid = (lo + hi) // 2
        out.append(mid)
        if nums[mid] == target:
            break
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return out, (lo, hi)

assert ans_ex1_mids == mids_closed([1, 4, 7, 10, 13, 16, 19], 16)[0]
assert ans_ex2_lo_hi == mids_closed([2, 4, 6], 5)[1]
assert ans_ex3_search([-1, 0, 3, 5, 9, 12], 9) == 4
assert ans_ex4_last_leq([1, 3, 5], 4) == 1 and ans_ex4_last_leq([1, 3, 5], 0) == -1
assert ans_ex5_first_last([5, 7, 7, 8, 8, 10], 8) == [3, 4]
assert ans_ex5_first_last([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert ans_ex6_search_rotated([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert ans_ex6_search_rotated([4, 5, 6, 7, 0, 1, 2], 3) == -1
for _ in range(300):
    arr = sorted(rng.sample(range(-40, 40), rng.randint(1, 20)))
    r = rng.randint(0, len(arr) - 1)
    rot = arr[r:] + arr[:r]
    for t in range(-42, 42):
        assert ans_ex6_search_rotated(rot, t) == (rot.index(t) if t in rot else -1)
        assert ans_ex4_last_leq(arr, t) == max([i for i, v in enumerate(arr) if v <= t], default=-1)
    dup = sorted(rng.choice(range(-5, 5)) for _ in range(rng.randint(1, 15)))
    for t in range(-6, 6):
        idx = [i for i, v in enumerate(dup) if v == t]
        assert ans_ex5_first_last(dup, t) == ([idx[0], idx[-1]] if idx else [-1, -1])
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


### Template nhớ nhanh (khoảng đóng)

```python
class Solution:
    def search(self, nums: List[int], target: int) -> int:
        lo, hi = 0, len(nums) - 1
        while lo <= hi:
            mid = (lo + hi) // 2
            if nums[mid] == target:
                return mid
            if nums[mid] < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1
```

Câu thần chú: **"Đóng–đóng thì `<=` và `±1`; nửa mở thì `<` và `hi = mid`."**

### Checklist trước khi submit
- [ ] Chọn một khuôn, ba chỗ (`hi` ban đầu, điều kiện `while`, cập nhật `hi`) khớp nhau.
- [ ] `mid = (lo + hi) // 2` dùng `//`.
- [ ] Cập nhật luôn loại `mid` (`mid + 1` / `mid - 1`) ở khoảng đóng → không vòng lặp vô hạn.
- [ ] Test mảng 1 phần tử, target ở đầu/cuối, nhỏ hơn/lớn hơn mọi phần tử.
- [ ] Time `O(log n)`, Space `O(1)`.